In [3]:

# @title xGLab JSON → XLSX

!pip install -q pandas openpyxl

import pandas as pd
import json
import os
import glob
import re
import sqlite3
import shutil

from pathlib import Path
from google.colab import drive


# ============================================================
# 1. НАСТРОЙКИ
# ============================================================

# Маска JSON-файлов:
# xg_rpl_*.json - Российская Премьер-лига
# xg_epl_*.json - Английская Премьер-лига

JSON_MASK = 'xg_rpl_*.json'

# Проверяем формат маски и определяем лигу
match = re.fullmatch(r'xg_(rpl|epl)_\*\.json', JSON_MASK)

if not match:
    raise ValueError(
        "Некорректная маска JSON_MASK. "
        "Допустимые варианты: "
        "'xg_rpl_*.json' или 'xg_epl_*.json'"
    )

league = match.group(1)

# Выбираем базу данных
DATABASES = {
    'rpl': 'rpl.db',
    'epl': 'FA_Premier_League.db'
}

DB_NAME = DATABASES[league]

# Google Drive
DRIVE_FOLDER = Path('/content/drive/MyDrive/Data_Bases')
DRIVE_DB_PATH = DRIVE_FOLDER / DB_NAME

# Локальная копия базы
LOCAL_DB_PATH = Path('/content') / DB_NAME

# Выходной Excel
output_xlsx_file = f'combined_xg_{league}_data.xlsx'


# ============================================================
# 2. ПОДКЛЮЧЕНИЕ GOOGLE DRIVE
# ============================================================

drive.mount('/content/drive')

if not DRIVE_DB_PATH.exists():
    raise FileNotFoundError(
        f"База данных не найдена: {DRIVE_DB_PATH}"
    )


print(f"Выбрана лига: {league.upper()}")
print(f"База данных: {DB_NAME}")
print(f"Маска JSON: {JSON_MASK}")


# ============================================================
# 3. ПОРЯДОК СТОЛБЦОВ
# ============================================================

final_output_columns_order = [
    "match_id",
    "player_name",
    "player_id",
    "minsPlayed",

    "epv_carry_n",
    "epv_carry_p",

    "epv_open_play_acc_n",
    "epv_open_play_acc_p",
    "epv_open_play_inacc_intercept",
    "epv_open_play_inacc_offside",
    "epv_open_play_inacc_out",

    "epv_set_piece_corner_acc_n",
    "epv_set_piece_corner_acc_p",
    "epv_set_piece_corner_inacc",

    "epv_set_piece_freekick_acc_n",
    "epv_set_piece_freekick_acc_p",
    "epv_set_piece_freekick_inacc",

    "epv_set_piece_throw_acc_n",
    "epv_set_piece_throw_acc_p",
    "epv_set_piece_throw_inacc"
]


# ============================================================
# 4. СОПОСТАВЛЕНИЕ СТОЛБЦОВ JSON И XLSX
# ============================================================

json_to_output_column_map = {

    "match_id": "match_id",
    "player_name": "player_name",
    "minsPlayed": "minsPlayed",

    "Carry_BETA.MINUS": "epv_carry_n",
    "Carry_BETA.PLUS": "epv_carry_p",

    "Open_Play.Accurate.MINUS": "epv_open_play_acc_n",
    "Open_Play.Accurate.PLUS": "epv_open_play_acc_p",

    "Open_Play.Inaccurate.Intercepted":
        "epv_open_play_inacc_intercept",

    "Open_Play.Inaccurate.To_Offside":
        "epv_open_play_inacc_offside",

    "Open_Play.Inaccurate.To_Out":
        "epv_open_play_inacc_out",

    "Set_Piece.Corner.Accurate_MINUS":
        "epv_set_piece_corner_acc_n",

    "Set_Piece.Corner.Accurate_PLUS":
        "epv_set_piece_corner_acc_p",

    "Set_Piece.Corner.Inaccurate":
        "epv_set_piece_corner_inacc",

    "Set_Piece.Free_Kick.Accurate_MINUS":
        "epv_set_piece_freekick_acc_n",

    "Set_Piece.Free_Kick.Accurate_PLUS":
        "epv_set_piece_freekick_acc_p",

    "Set_Piece.Free_Kick.Inaccurate":
        "epv_set_piece_freekick_inacc",

    "Set_Piece.Throw_In.Accurate_MINUS":
        "epv_set_piece_throw_acc_n",

    "Set_Piece.Throw_In.Accurate_PLUS":
        "epv_set_piece_throw_acc_p",

    "Set_Piece.Throw_In.Inaccurate":
        "epv_set_piece_throw_inacc"
}


# ============================================================
# 5. ЗАГРУЗКА ИМЁН И ID ИЗ SQLITE
# ============================================================

def load_player_mapping(db_path):

    conn = sqlite3.connect(db_path)

    try:

        query = """
            SELECT
                player_id,
                xglab_name
            FROM Players
            WHERE xglab_name IS NOT NULL
              AND TRIM(xglab_name) != ''
        """

        players_df = pd.read_sql_query(query, conn)

    finally:
        conn.close()

    # Нормализуем имена для сопоставления
    players_df['normalized_name'] = (
        players_df['xglab_name']
        .astype(str)
        .str.strip()
        .str.casefold()
    )

    # Проверяем неоднозначные имена
    name_counts = (
        players_df.groupby('normalized_name')['player_id']
        .nunique()
    )

    ambiguous_names = set(
        name_counts[name_counts > 1].index
    )

    # Создаём словарь только для однозначных совпадений
    unique_players = players_df[
        ~players_df['normalized_name'].isin(ambiguous_names)
    ]

    player_mapping = dict(
        zip(
            unique_players['normalized_name'],
            unique_players['player_id']
        )
    )

    print("\nЗагрузка игроков из базы данных:")
    print(f"Всего записей с xglab_name: {len(players_df)}")
    print(f"Однозначных имён: {len(player_mapping)}")
    print(f"Неоднозначных имён: {len(ambiguous_names)}")

    if ambiguous_names:
        print("\nНеоднозначные имена в базе:")
        for name in sorted(ambiguous_names):
            print(f"  - {name}")

    return player_mapping


player_mapping = load_player_mapping(DRIVE_DB_PATH)


# ============================================================
# 6. ПОИСК JSON-ФАЙЛОВ
# ============================================================

json_files = sorted(glob.glob(JSON_MASK))

if not json_files:

    raise FileNotFoundError(
        f"JSON-файлы по маске '{JSON_MASK}' "
        "не найдены в текущей директории."
    )

print(f"\nНайдено JSON-файлов: {len(json_files)}")


# ============================================================
# 7. ОБРАБОТКА JSON
# ============================================================

all_processed_data = []

file_error_count = 0
total_records = 0

for input_json_file in json_files:

    try:

        print(f"\nОбработка файла: {input_json_file}")

        # Извлекаем match_id из имени файла
        filename = os.path.basename(input_json_file)

        filename_match = re.fullmatch(
            r'xg_(?:rpl|epl)_(.+)\.json',
            filename
        )

        if not filename_match:

            print(
                f"Предупреждение: некорректное имя файла "
                f"{filename}. Файл пропущен."
            )

            file_error_count += 1
            continue

        match_id = filename_match.group(1)

        # Читаем JSON
        with open(
            input_json_file,
            'r',
            encoding='utf-8'
        ) as f:

            json_data = json.load(f)

        # Проверяем структуру
        if (
            not isinstance(json_data, dict)
            or 'records' not in json_data
            or not isinstance(json_data['records'], list)
        ):

            print(
                f"Предупреждение: неподдерживаемая структура "
                f"JSON в файле {filename}."
            )

            file_error_count += 1
            continue

        records = json_data['records']

        file_records = 0

        for record in records:

            player_name = record.get('playerName')

            # Подготавливаем имя для сопоставления
            normalized_name = (
                str(player_name).strip().casefold()
                if player_name is not None
                else None
            )

            # Получаем player_id
            player_id = (
                player_mapping.get(normalized_name)
                if normalized_name
                else None
            )

            player_data = {

                "match_id": match_id,
                "player_id": player_id,
                "player_name": player_name,
                "minsPlayed": record.get("minsPlayed")
            }

            # Извлекаем классы JSON
            for class_item in record.get("classes", []):

                class_name = class_item.get("className")
                value = class_item.get("value")

                if class_name:
                    player_data[class_name] = value

            all_processed_data.append(player_data)

            file_records += 1
            total_records += 1

        print(
            f"Обработано записей: {file_records}"
        )

    except Exception as e:

        print(
            f"Ошибка обработки файла {input_json_file}: {e}"
        )

        file_error_count += 1


# ============================================================
# 8. ФОРМИРОВАНИЕ DATAFRAME
# ============================================================

if not all_processed_data:

    raise ValueError(
        "Не удалось обработать данные ни из одного JSON-файла."
    )

df = pd.DataFrame(all_processed_data)


# ============================================================
# 9. ПЕРЕИМЕНОВАНИЕ СТОЛБЦОВ
# ============================================================

present_json_cols_to_rename = {

    json_col_key: output_col_name

    for json_col_key, output_col_name
    in json_to_output_column_map.items()

    if json_col_key in df.columns
}

df.rename(
    columns=present_json_cols_to_rename,
    inplace=True
)


# ============================================================
# 10. ОКРУГЛЕНИЕ ЧИСЛОВЫХ ПАРАМЕТРОВ
# ============================================================

# Исключения:
# match_id - идентификатор матча
# player_id - идентификатор игрока
# player_name - имя игрока
# minsPlayed - минуты сыгранного времени

columns_not_to_round = {
    'match_id',
    'player_id',
    'player_name',
    'minsPlayed'
}

for column in df.columns:

    if column in columns_not_to_round:
        continue

    # Преобразуем числовые значения и округляем до 3 знаков
    numeric_values = pd.to_numeric(
        df[column],
        errors='coerce'
    )

    # Если в столбце есть числовые значения,
    # округляем их, сохраняя исходные нечисловые значения
    numeric_mask = numeric_values.notna()

    df.loc[numeric_mask, column] = (
        numeric_values[numeric_mask].round(3)
    )


# ============================================================
# 11. ПРОВЕРКА СОПОСТАВЛЕНИЯ ИГРОКОВ
# ============================================================

unmatched_players = df[
    df['player_id'].isna()
]['player_name'].dropna().unique()

matched_count = df['player_id'].notna().sum()
unmatched_count = df['player_id'].isna().sum()

print("\n" + "=" * 60)
print("РЕЗУЛЬТАТ СОПОСТАВЛЕНИЯ ИГРОКОВ")
print("=" * 60)

print(f"Всего записей: {len(df)}")
print(f"Найдено player_id: {matched_count}")
print(f"Не найдено player_id: {unmatched_count}")

if len(df) > 0:

    match_rate = matched_count / len(df) * 100

    print(f"Успешность сопоставления: {match_rate:.2f}%")


if len(unmatched_players) > 0:

    print("\nИгроки без найденного player_id:")

    for player in sorted(unmatched_players):
        print(f"  - {player}")


# ============================================================
# 12. ФОРМИРОВАНИЕ ИТОГОВОГО XLSX
# ============================================================

# Добавляем отсутствующие столбцы
for column in final_output_columns_order:

    if column not in df.columns:
        df[column] = pd.NA

# Устанавливаем требуемый порядок
df_filtered = df[final_output_columns_order].copy()

# Сохраняем Excel
df_filtered.to_excel(
    output_xlsx_file,
    index=False
)


# ============================================================
# 13. ИТОГОВЫЙ ОТЧЁТ
# ============================================================

print("\n" + "=" * 60)
print("ПРЕОБРАЗОВАНИЕ JSON → XLSX ЗАВЕРШЕНО")
print("=" * 60)

print(f"Лига: {league.upper()}")
print(f"База данных: {DB_NAME}")
print(f"Маска файлов: {JSON_MASK}")
print(f"Обработано JSON-файлов: {len(json_files)}")
print(f"Ошибок обработки файлов: {file_error_count}")
print(f"Всего записей: {total_records}")
print(f"Успешных сопоставлений: {matched_count}")
print(f"Несопоставленных записей: {unmatched_count}")
print(f"Итоговый файл: {output_xlsx_file}")

print("=" * 60)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Выбрана лига: RPL
База данных: rpl.db
Маска JSON: xg_rpl_*.json

Загрузка игроков из базы данных:
Всего записей с xglab_name: 739
Однозначных имён: 739
Неоднозначных имён: 0

Найдено JSON-файлов: 8

Обработка файла: xg_rpl_585.json
Обработано записей: 29

Обработка файла: xg_rpl_586.json
Обработано записей: 30

Обработка файла: xg_rpl_587.json
Обработано записей: 30

Обработка файла: xg_rpl_588.json
Обработано записей: 30

Обработка файла: xg_rpl_589.json
Обработано записей: 31

Обработка файла: xg_rpl_590.json
Обработано записей: 31

Обработка файла: xg_rpl_591.json
Обработано записей: 28

Обработка файла: xg_rpl_592.json
Обработано записей: 28

РЕЗУЛЬТАТ СОПОСТАВЛЕНИЯ ИГРОКОВ
Всего записей: 237
Найдено player_id: 234
Не найдено player_id: 3
Успешность сопоставления: 98.73%

Игроки без найденного player_id:
  - Roberto Fernández
  - Sergey Volkov

ПРЕОБРАЗОВ

In [4]:
# @title PlayerStats JSON → XLSX

import pandas as pd
import sqlite3
import os
import re
import math
from collections import Counter
from pathlib import Path
from google.colab import drive

# ============================================================
# 1. ПОДКЛЮЧЕНИЕ GOOGLE DRIVE И ВЫБОР ЛИГИ
# ============================================================

drive.mount('/content/drive')

# Меняйте только эту маску:
# 'ps_epl_*.json' или 'ps_rpl_*.json'
json_file_mask = 'ps_rpl_*.json'

league_search = re.fullmatch(r'ps_([er]pl)_\*\.json', json_file_mask)

if not league_search:
    raise ValueError(
        "Используйте маску 'ps_epl_*.json' или 'ps_rpl_*.json'."
    )

league = league_search.group(1)

# Имя итогового XLSX
excel_file_name = f'PlayerStats{league.upper()}27.xlsx'

# Файл xGLab, созданный предыдущим скриптом
xglab_file_name = f'combined_xg_{league}_data.xlsx'

# SQLite
db_file_name = (
    'FA_Premier_League.db'
    if league == 'epl'
    else 'rpl.db'
)

db_path = Path('/content/drive/MyDrive/Data_Bases') / db_file_name

# Маска JSON
json_file_pattern = re.compile(
    re.escape(json_file_mask).replace(r'\*', r'(\d+)')
)

print('=' * 65)
print('ЗАПУСК ОБРАБОТКИ PLAYERSTATS')
print('=' * 65)
print(f'Лига: {league.upper()}')
print(f'Маска JSON: {json_file_mask}')
print(f'База данных: {db_path}')
print(f'Файл xGLab: {xglab_file_name}')
print(f'Итоговый файл: {excel_file_name}')


# ============================================================
# 2. ПРОВЕРКА ФАЙЛОВ
# ============================================================

if not db_path.is_file():
    raise FileNotFoundError(
        f'База данных не найдена: {db_path}'
    )

if not os.path.isfile(xglab_file_name):
    raise FileNotFoundError(
        f"Не найден XLSX предыдущего скрипта xGLab: "
        f"'{xglab_file_name}'. "
        f"Сначала выполните скрипт xGLab."
    )

json_files_to_process = sorted([
    f for f in os.listdir('.')
    if json_file_pattern.fullmatch(f)
    and os.path.isfile(f)
])

if not json_files_to_process:
    raise FileNotFoundError(
        f"JSON-файлы по маске '{json_file_mask}' не найдены."
    )

print(f'\nНайдено JSON-файлов: {len(json_files_to_process)}')


# ============================================================
# 3. ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ============================================================

def normalize_name(value):
    """Нормализация имени для сопоставления."""
    if pd.isna(value):
        return None

    value = str(value).strip().casefold()

    return value if value else None


def normalize_id(value):
    """Приведение ID к единому строковому формату."""
    if pd.isna(value):
        return None

    try:
        numeric_value = float(value)

        if numeric_value.is_integer():
            return str(int(numeric_value))

    except (ValueError, TypeError):
        pass

    value = str(value).strip()

    return value if value else None


def flatten_player_data(player_list):
    """Разворачивание вложенных данных игроков."""
    flat_data = []

    for player_dict in player_list:
        flattened_player = player_dict.copy()

        if 'player' in flattened_player:
            player_details = flattened_player.pop('player')
            flattened_player.update(player_details)

        flat_data.append(flattened_player)

    return flat_data


# ============================================================
# 4. ЗАГРУЗКА СПРАВОЧНИКА ИГРОКОВ И MAX PLAYER_STAT_ID
# ============================================================

print('\n' + '=' * 65)
print('ПОДКЛЮЧЕНИЕ К SQLITE')
print('=' * 65)

# Режим mode=ro гарантирует открытие базы только для чтения.
db_uri = f'{db_path.resolve().as_uri()}?mode=ro'

with sqlite3.connect(db_uri, uri=True) as conn:

    players_reference_df = pd.read_sql_query(
        """
        SELECT
            player_id,
            sofascore_name
        FROM Players
        """,
        conn
    )

    max_id_result = pd.read_sql_query(
        """
        SELECT MAX(player_stat_id) AS max_id
        FROM PlayerStats
        """,
        conn
    )

    max_player_stat_id = max_id_result.loc[0, 'max_id']

    if pd.isna(max_player_stat_id):
        first_player_stat_id = 1
    else:
        first_player_stat_id = int(max_player_stat_id) + 1


print(f'Игроков в таблице Players: {len(players_reference_df)}')
print(f'Последний player_stat_id в базе: {max_player_stat_id}')
print(f'Первый новый player_stat_id: {first_player_stat_id}')


# ============================================================
# 5. СОЗДАНИЕ СЛОВАРЯ PLAYER_ID
# ============================================================

players_reference_df['normalized_name'] = (
    players_reference_df['sofascore_name'].apply(normalize_name)
)

players_reference_df['normalized_player_id'] = (
    players_reference_df['player_id'].apply(normalize_id)
)

players_reference_df = players_reference_df.dropna(
    subset=['normalized_name', 'normalized_player_id']
)

# Аналогично xGLab:
# - одинаковое имя с одним player_id допустимо;
# - одинаковое имя с разными player_id считается неоднозначным;
# - неоднозначные имена не используются.

player_id_mapping = {}
ambiguous_player_names = []

for player_name, group in players_reference_df.groupby(
    'normalized_name'
):

    unique_ids = group['normalized_player_id'].unique()

    if len(unique_ids) == 1:
        player_id_mapping[player_name] = unique_ids[0]
    else:
        ambiguous_player_names.append(player_name)

print(f'Уникальных имён для сопоставления: {len(player_id_mapping)}')
print(f'Неоднозначных имён в Players: {len(ambiguous_player_names)}')


# ============================================================
# 6. ЗАГРУЗКА МИНУТ ИЗ XLSX XGLAB
# ============================================================

print('\n' + '=' * 65)
print('ЗАГРУЗКА МИНУТ ИЗ XLSX XGLAB')
print('=' * 65)

xglab_df = pd.read_excel(
    xglab_file_name,
    engine='openpyxl'
)

required_xglab_columns = [
    'match_id',
    'player_id',
    'minsPlayed'
]

missing_columns = [
    col for col in required_xglab_columns
    if col not in xglab_df.columns
]

if missing_columns:
    raise ValueError(
        f'В файле {xglab_file_name} отсутствуют столбцы: '
        f'{missing_columns}'
    )

xglab_df['match_key'] = xglab_df['match_id'].apply(normalize_id)
xglab_df['player_key'] = xglab_df['player_id'].apply(normalize_id)

xglab_df['minutes_value'] = pd.to_numeric(
    xglab_df['minsPlayed'],
    errors='coerce'
)

xglab_df = xglab_df.dropna(
    subset=['match_key', 'player_key', 'minutes_value']
)

# Ключ сопоставления — match_id + player_id.
# Это позволяет получить минуты именно из соответствующего матча.

minutes_mapping = {}
ambiguous_minutes_keys = []

for (match_key, player_key), group in xglab_df.groupby(
    ['match_key', 'player_key']
):

    unique_minutes = group['minutes_value'].unique()

    if len(unique_minutes) == 1:
        minutes_mapping[(match_key, player_key)] = unique_minutes[0]
    else:
        ambiguous_minutes_keys.append((match_key, player_key))

print(f'Строк в XLSX xGLab: {len(xglab_df)}')
print(f'Уникальных пар match_id + player_id: {len(minutes_mapping)}')
print(f'Неоднозначных пар по минутам: {len(ambiguous_minutes_keys)}')


# ============================================================
# 7. СООТВЕТСТВИЕ JSON → XLSX
# ============================================================

column_mapping = {
    'name': 'plsyer_name',
    'minutesPlayed': 'minutes_played',
    'goals': 'goals',
    'goalAssist': 'assists',
    'totalClearance': 'clearances',
    'ballRecovery': 'recovery',
    'outfielderBlock': 'blocked_shots',
    'interceptionWon': 'interceptions',
    'totalTackle': 'total_tackles',
    'wonTackle': 'tackles_won',
    'challengeLost': 'dribbled_past',
    'duelLost': 'ground_duels',
    'duelWon': 'ground_duels_won',
    'aerialLost': 'aerial_duels',
    'aerialWon': 'aerial_duels_won',
    'possessionLostCtrl': 'possession_lost',
    'fouls': 'fouls',
    'wasFouled': 'was_fouled',
    'touches': 'touches',
    'unsuccessfulTouch': 'touches_unsuc',
    'totalPass': 'passes_total',
    'accuratePass': 'passes_accurate',
    'totalOppositionHalfPasses': 'passes_in_opp_total',
    'accurateOppositionHalfPasses': 'passes_in_opp_accurate',
    'totalOwnHalfPasses': 'passes_in_own_total',
    'accurateOwnHalfPasses': 'passes_in_own_accurate',
    'keyPass': 'key_passes',
    'totalCross': 'crosses_total',
    'accurateCross': 'crosses_accurate',
    'totalLongBalls': 'long_balls_total',
    'accurateLongBalls': 'long_balls_accurate',
    'onTargetScoringAttempt': 'shots_on_target',
    'shotOffTarget': 'shots_off_target',
    'blockedScoringAttempt': 'shots_blocked',
    'totalContest': 'dribble_att',
    'wonContest': 'dribble_succ',
    'rating': 'rating',
    'bigChanceCreated': 'big_chances_created',
    'bigChanceMissed': 'big_chances_missed',
    'totalOffside': 'offsides',
    'hitWoodwork': 'hit_woodwork',
    'clearanceOffLine': 'clearance_off_line',
    'ownGoals': 'own_goals',
    'errorLeadToAShot': 'errors_lead_to_shot',
    'errorLeadToAGoal': 'errors_lead_to_goal',
    'penaltyWon': 'penalty_won',
    'penaltyConceded': 'penalty_committed',
    'penaltyMiss': 'penalty_unrealized',
    'kilometersCovered': 'distance_covered',
    'topSpeed': 'top_speed',
    'numberOfSprints': 'number_of_sprints',
    'metersCoveredRunningKm': 'running',
    'metersCoveredHighSpeedRunningKm': 'high_speed_running',
    'metersCoveredSprintingKm': 'sprinting',
    'totalBallCarriesDistance': 'total_carrying_distance',
    'ballCarriesCount': 'carries',
    'progressiveBallCarriesCount': 'progressive_carries',
    'totalProgression': 'total_progression',
    'totalProgressiveBallCarriesDistance': 'progressive_carrying_distance',
    'bestBallCarryProgression': 'longest_progressive_carry',
    'substitute': 'is_substitute',
    'teamId': 'team_id'
}


# ============================================================
# 8. ПОРЯДОК СТОЛБЦОВ
# ============================================================

final_xlsx_columns_order = [
    'match_id',
    'player_stat_id',
    'plsyer_name',
    'player_id',
    'team_id',
    'minutes_played',
    'goals',
    'assists',
    'yellow_card',
    'second_yellow_card',
    'red_card',
    'xg',
    'xa',
    'xGChain',
    'xGBuildup',
    'clearances',
    'recovery',
    'blocked_shots',
    'interceptions',
    'total_tackles',
    'tackles_won',
    'dribbled_past',
    'ground_duels',
    'ground_duels_won',
    'aerial_duels',
    'aerial_duels_won',
    'possession_lost',
    'fouls',
    'was_fouled',
    'touches',
    'touches_unsuc',
    'passes_total',
    'passes_accurate',
    'passes_in_opp_total',
    'passes_in_opp_accurate',
    'passes_in_own_total',
    'passes_in_own_accurate',
    'key_passes',
    'crosses_total',
    'crosses_accurate',
    'long_balls_total',
    'long_balls_accurate',
    'shots_on_target',
    'shots_off_target',
    'shots_blocked',
    'dribble_att',
    'dribble_succ',
    'starting_player',
    'position_on_field',
    'rating',
    'big_chances_created',
    'big_chances_missed',
    'offsides',
    'hit_woodwork',
    'clearance_off_line',
    'own_goals',
    'errors_lead_to_shot',
    'errors_lead_to_goal',
    'penalty_won',
    'penalty_committed',
    'penalty_realized',
    'penalty_unrealized',
    'distance_covered',
    'top_speed',
    'number_of_sprints',
    'running',
    'high_speed_running',
    'sprinting',
    'total_carrying_distance',
    'carries',
    'progressive_carries',
    'total_progression',
    'progressive_carrying_distance',
    'longest_progressive_carry'
]


# ============================================================
# 9. ЗАМЕНА TEAM_ID
# ============================================================

team_id_mapping = {
    42: 2,
    34: 18,
    40: 9,
    50: 13,
    38: 3,
    60: 11,
    32: 22,
    43: 15,
    41: 19,
    30: 8,
    35: 6,
    33: 5,
    7: 10,
    14: 12,
    96: 23,
    48: 17,
    44: 4,
    17: 1,
    11: 21,
    39: 7,
    2326: 7,
    285689: 9,
    24118: 16,
    362016: 13,
    2317: 15,
    2320: 6,
    2322: 10,
    2323: 4,
    34425: 3,
    2321: 2,
    5131: 14,
    7517: 17,
    2325: 1,
    322699: 20,
    2333: 8,
    2315: 5
}


# ============================================================
# 10. СТОЛБЦЫ ДЛЯ ОКРУГЛЕНИЯ
# ============================================================

round_columns = [
    'distance_covered',
    'top_speed',
    'running',
    'high_speed_running',
    'sprinting',
    'total_carrying_distance',
    'carries',
    'progressive_carries',
    'total_progression',
    'progressive_carrying_distance',
    'longest_progressive_carry'
]


# ============================================================
# 11. СТАТИСТИКА РАБОТЫ СКРИПТА
# ============================================================

all_matches_data = []

report = {
    'files_found': len(json_files_to_process),
    'files_processed': 0,
    'files_with_errors': 0,
    'files_without_players': 0,
    'players_before_filter': 0,
    'players_after_filter': 0,
    'player_id_found': 0,
    'player_id_not_found': 0,
    'minutes_found': 0,
    'minutes_not_found': 0,
    'ambiguous_player_names': 0,
    'ambiguous_minutes': 0
}

unmatched_player_names = Counter()
unmatched_minutes = Counter()
file_errors = []


# ============================================================
# 12. ОБРАБОТКА JSON
# ============================================================

for json_file in json_files_to_process:

    print(f'\nОбработка: {json_file}')

    match_id_search = json_file_pattern.fullmatch(json_file)

    if not match_id_search:
        print('Не удалось извлечь match_id. Файл пропущен.')
        continue

    match_id = int(match_id_search.group(1))

    try:

        with open(json_file, 'r', encoding='utf-8') as f:
            df_raw = pd.read_json(f)

        home_players_data = df_raw.loc['players', 'home']
        away_players_data = df_raw.loc['players', 'away']

        home_flat_players = flatten_player_data(home_players_data)
        away_flat_players = flatten_player_data(away_players_data)

        df_home_players = pd.DataFrame(home_flat_players)
        df_away_players = pd.DataFrame(away_flat_players)

        df_home_players['team_category'] = 'Home'
        df_away_players['team_category'] = 'Away'

        match_df = pd.concat(
            [df_home_players, df_away_players],
            ignore_index=True
        )

        if match_df.empty:
            report['files_without_players'] += 1
            print('В файле нет игроков.')
            continue

        match_df['match_id'] = match_id

        # Разворачиваем статистику
        if 'statistics' in match_df.columns:

            stats_df = pd.json_normalize(
                match_df['statistics'].apply(
                    lambda x: x if isinstance(x, dict) else {}
                )
            )

            match_df = match_df.drop(columns=['statistics'])

            match_df = pd.concat(
                [match_df.reset_index(drop=True), stats_df],
                axis=1
            )

        # Переименовываем столбцы
        match_df = match_df.rename(columns=column_mapping)

        # Стартовый состав / запасные
        if 'is_substitute' in match_df.columns:

            match_df['starting_player'] = (
                match_df['is_substitute'].apply(
                    lambda x: 0 if pd.isna(x) else (0 if x else 1)
                )
            )

            match_df = match_df.drop(columns=['is_substitute'])

        else:
            match_df['starting_player'] = 0

        # Добавляем отсутствующие столбцы
        for col in final_xlsx_columns_order:

            if col not in match_df.columns:
                match_df[col] = pd.NA

        # ----------------------------------------------------
        # PLAYER_ID: СОПОСТАВЛЕНИЕ ЧЕРЕЗ SOFASCORE_NAME
        # ----------------------------------------------------

        match_df['normalized_name'] = (
            match_df['plsyer_name'].apply(normalize_name)
        )

        match_df['player_id'] = (
            match_df['normalized_name'].map(player_id_mapping)
        )

        # Статистика по player_id
        report['players_before_filter'] += len(match_df)

        found_ids = match_df['player_id'].notna()

        report['player_id_found'] += int(found_ids.sum())
        report['player_id_not_found'] += int((~found_ids).sum())

        for name in match_df.loc[
            ~found_ids, 'plsyer_name'
        ].dropna():

            unmatched_player_names[str(name)] += 1

        # ----------------------------------------------------
        # ЧИСЛОВЫЕ ПОЛЯ
        # ----------------------------------------------------

        for col in final_xlsx_columns_order:

            if col in [
                'match_id',
                'player_stat_id',
                'plsyer_name',
                'player_id',
                'position_on_field',
                'minutes_played',
                'xg',
                'xa',
                'xGChain',
                'xGBuildup'
            ]:
                continue

            match_df[col] = pd.to_numeric(
                match_df[col],
                errors='coerce'
            )

            if col != 'rating':
                match_df[col] = match_df[col].fillna(0)

        # ----------------------------------------------------
        # ФИЛЬТРАЦИЯ ИГРОКОВ ПО МИНУТАМ ИЗ JSON
        # ----------------------------------------------------
        # Сначала используем minutesPlayed из JSON только для
        # удаления игроков, не выходивших на поле.
        # Затем значение будет заменено минутами из xGLab.

        json_minutes = pd.to_numeric(
            match_df['minutes_played'],
            errors='coerce'
        )

        match_df = match_df[json_minutes > 0].copy()

        report['players_after_filter'] += len(match_df)

        # ----------------------------------------------------
        # MINUTES_PLAYED: ЗАГРУЗКА ИЗ XGLAB
        # ----------------------------------------------------

        match_df['match_key'] = match_df['match_id'].apply(
            normalize_id
        )

        match_df['player_key'] = match_df['player_id'].apply(
            normalize_id
        )

        def get_minutes(row):

            match_key = row['match_key']
            player_key = row['player_key']

            if not match_key or not player_key:
                return pd.NA

            key = (match_key, player_key)

            if key in minutes_mapping:
                return minutes_mapping[key]

            return pd.NA

        match_df['minutes_played'] = match_df.apply(
            get_minutes,
            axis=1
        )

        minutes_found_mask = match_df['minutes_played'].notna()

        report['minutes_found'] += int(minutes_found_mask.sum())
        report['minutes_not_found'] += int(
            (~minutes_found_mask).sum()
        )

        for _, row in match_df.loc[
            ~minutes_found_mask
        ].iterrows():

            unmatched_minutes[
                f"match_id={row['match_id']}, "
                f"player_id={row['player_id']}, "
                f"name={row['plsyer_name']}"
            ] += 1

        # Удаляем технические столбцы
        match_df = match_df.drop(
            columns=['normalized_name', 'match_key', 'player_key']
        )

        # ----------------------------------------------------
        # ДУЭЛИ
        # ----------------------------------------------------

        match_df['ground_duels'] = (
            match_df['ground_duels']
            + match_df['ground_duels_won']
        )

        match_df['aerial_duels'] = (
            match_df['aerial_duels']
            + match_df['aerial_duels_won']
        )

        # ----------------------------------------------------
        # TEAM_ID
        # ----------------------------------------------------

        match_df['team_id'] = match_df['team_id'].replace(
            team_id_mapping
        )

        # ----------------------------------------------------
        # ПУСТЫЕ ПОЛЯ
        # ----------------------------------------------------

        for col in [
            'xg',
            'xa',
            'xGChain',
            'xGBuildup'
        ]:
            match_df[col] = pd.NA

        # ----------------------------------------------------
        # POSITION_ON_FIELD
        # ----------------------------------------------------

        match_df['position_on_field'] = pd.Series(
            pd.NA,
            index=match_df.index,
            dtype='object'
        )

        match_df.loc[
            match_df['starting_player'] == 0,
            'position_on_field'
        ] = 'Sub'

        # ----------------------------------------------------
        # ОКРУГЛЕНИЕ
        # ----------------------------------------------------

        match_df[round_columns] = (
            match_df[round_columns].round(2)
        )

        all_matches_data.append(match_df)

        report['files_processed'] += 1

        print(f'Игроков после фильтрации: {len(match_df)}')
        print(
            f'Найдено player_id: '
            f'{int(found_ids.sum())}'
        )
        print(
            f'Найдено минут: '
            f'{int(minutes_found_mask.sum())}'
        )

    except Exception as e:

        report['files_with_errors'] += 1

        file_errors.append(
            f'{json_file}: {type(e).__name__}: {e}'
        )

        print(f'ОШИБКА: {type(e).__name__}: {e}')


# ============================================================
# 13. ОБЪЕДИНЕНИЕ И ФОРМИРОВАНИЕ ИТОГОВОГО XLSX
# ============================================================

if not all_matches_data:

    print('\nДанные не были обработаны ни из одного JSON-файла.')

else:

    final_output_df = pd.concat(
        all_matches_data,
        ignore_index=True
    )

    # --------------------------------------------------------
    # СОРТИРОВКА: MATCH_ID → HOME → AWAY
    # --------------------------------------------------------

    final_output_df['match_id'] = pd.to_numeric(
        final_output_df['match_id'],
        errors='coerce'
    )

    final_output_df['team_category'] = pd.Categorical(
        final_output_df['team_category'],
        categories=['Home', 'Away'],
        ordered=True
    )

    final_output_df = final_output_df.sort_values(
        by=['match_id', 'team_category'],
        ascending=[True, True],
        kind='stable'
    ).reset_index(drop=True)

    final_output_df = final_output_df.drop(
        columns=['team_category']
    )

    # --------------------------------------------------------
    # PLAYER_STAT_ID
    # --------------------------------------------------------
    # Первый ID = MAX(player_stat_id) + 1 из SQLite.
    # Далее ID увеличивается на единицу для каждой строки.

    final_output_df['player_stat_id'] = range(
        first_player_stat_id,
        first_player_stat_id + len(final_output_df)
    )

    # Финальный порядок столбцов
    final_output_df = final_output_df[
        final_xlsx_columns_order
    ]

    # --------------------------------------------------------
    # СОХРАНЕНИЕ XLSX
    # --------------------------------------------------------

    with pd.ExcelWriter(
        excel_file_name,
        engine='openpyxl'
    ) as writer:

        final_output_df.to_excel(
            writer,
            index=False,
            sheet_name='Sheet1',
            na_rep=''
        )

        worksheet = writer.sheets['Sheet1']

        # Формат числовых столбцов
        for col in round_columns:

            column_index = (
                final_output_df.columns.get_loc(col) + 1
            )

            for cells in worksheet.iter_cols(
                min_col=column_index,
                max_col=column_index,
                min_row=2,
                max_row=len(final_output_df) + 1
            ):

                for cell in cells:
                    cell.number_format = '0.00'


    # ========================================================
    # 14. ФИНАЛЬНЫЙ ДЕТАЛЬНЫЙ ОТЧЁТ
    # ========================================================

    report['ambiguous_player_names'] = len(
        ambiguous_player_names
    )

    report['ambiguous_minutes'] = len(
        ambiguous_minutes_keys
    )

    print('\n')
    print('=' * 70)
    print('ДЕТАЛЬНЫЙ ОТЧЁТ О РАБОТЕ СКРИПТА PLAYERSTATS')
    print('=' * 70)

    print('\n1. ОБРАБОТКА ФАЙЛОВ')
    print('-' * 70)
    print(f"Найдено JSON-файлов: {report['files_found']}")
    print(f"Успешно обработано: {report['files_processed']}")
    print(f"Файлов с ошибками: {report['files_with_errors']}")
    print(f"Файлов без игроков: {report['files_without_players']}")

    print('\n2. ОБРАБОТКА ИГРОКОВ')
    print('-' * 70)
    print(
        f"Игроков до фильтрации: "
        f"{report['players_before_filter']}"
    )
    print(
        f"Игроков после фильтрации: "
        f"{report['players_after_filter']}"
    )

    print('\n3. СОПОСТАВЛЕНИЕ PLAYER_ID')
    print('-' * 70)
    print(
        f"Найдено player_id: "
        f"{report['player_id_found']}"
    )
    print(
        f"Не найдено player_id: "
        f"{report['player_id_not_found']}"
    )
    print(
        f"Неоднозначных имён в Players: "
        f"{report['ambiguous_player_names']}"
    )

    if unmatched_player_names:

        print('\nИгроки, для которых не найден player_id:')

        for name, count in unmatched_player_names.most_common():
            print(f'  {name}: {count}')

    if ambiguous_player_names:

        print('\nНеоднозначные имена в таблице Players:')

        for name in ambiguous_player_names:
            print(f'  {name}')

    print('\n4. СОПОСТАВЛЕНИЕ MINUTES_PLAYED')
    print('-' * 70)
    print(f'Источник: {xglab_file_name}')
    print('Столбец источника: minsPlayed')
    print('Сопоставление: match_id + player_id')
    print(
        f"Успешно заполнено минут: "
        f"{report['minutes_found']}"
    )
    print(
        f"Не удалось заполнить минут: "
        f"{report['minutes_not_found']}"
    )
    print(
        f"Неоднозначных пар в XLSX xGLab: "
        f"{report['ambiguous_minutes']}"
    )

    if unmatched_minutes:

        print('\nНе найдены минуты для следующих записей:')

        for key, count in unmatched_minutes.most_common():
            print(f'  {key}: {count}')

    if ambiguous_minutes_keys:

        print('\nНеоднозначные пары match_id + player_id:')

        for match_key, player_key in ambiguous_minutes_keys:
            print(
                f'  match_id={match_key}, '
                f'player_id={player_key}'
            )

    print('\n5. PLAYER_STAT_ID')
    print('-' * 70)
    print(f'Последний ID в SQLite: {max_player_stat_id}')
    print(f'Первый ID в XLSX: {first_player_stat_id}')

    if not final_output_df.empty:

        print(
            f"Последний ID в XLSX: "
            f"{final_output_df['player_stat_id'].iloc[-1]}"
        )

    print(
        f'Всего назначено player_stat_id: '
        f'{len(final_output_df)}'
    )

    print('\n6. ИТОГОВЫЙ ФАЙЛ')
    print('-' * 70)
    print(f'Файл: {excel_file_name}')
    print(f'Строк: {len(final_output_df)}')
    print(f'Столбцов: {len(final_output_df.columns)}')

    print('\n7. ОШИБКИ')
    print('-' * 70)

    if file_errors:

        for error in file_errors:
            print(f'  {error}')

    else:
        print('Ошибок при обработке JSON не обнаружено.')

    print('\n' + '=' * 70)
    print('ОБРАБОТКА ЗАВЕРШЕНА')
    print('=' * 70)
    print(
        f"Итоговый файл сохранён: {excel_file_name}"
    )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ЗАПУСК ОБРАБОТКИ PLAYERSTATS
Лига: RPL
Маска JSON: ps_rpl_*.json
База данных: /content/drive/MyDrive/Data_Bases/rpl.db
Файл xGLab: combined_xg_rpl_data.xlsx
Итоговый файл: PlayerStatsRPL27.xlsx

Найдено JSON-файлов: 8

ПОДКЛЮЧЕНИЕ К SQLITE
Игроков в таблице Players: 743
Последний player_stat_id в базе: 18148
Первый новый player_stat_id: 18149
Уникальных имён для сопоставления: 739
Неоднозначных имён в Players: 0

ЗАГРУЗКА МИНУТ ИЗ XLSX XGLAB
Строк в XLSX xGLab: 234
Уникальных пар match_id + player_id: 234
Неоднозначных пар по минутам: 0

Обработка: ps_rpl_585.json
Игроков после фильтрации: 30
Найдено player_id: 45
Найдено минут: 29

Обработка: ps_rpl_586.json
Игроков после фильтрации: 30
Найдено player_id: 44
Найдено минут: 29

Обработка: ps_rpl_587.json
Игроков после фильтрации: 30
Найдено player_id: 42
Найдено минут: 29

Обработка: ps_rpl_588.json
Игроков п

In [ ]:
# @title
import json
import re
import sqlite3
from pathlib import Path
import pandas as pd
from google.colab import drive

# Выберите маску файлов
JSON_MASK = 'ms_epl_*.json'  # или 'ms_epl_*.json'
match = re.fullmatch(r'ms_(rpl|epl)_\*\.json', JSON_MASK)
if not match:
    raise ValueError("JSON_MASK должен быть 'ms_rpl_*.json' или 'ms_epl_*.json'.")
league = match.group(1)
OUTPUT = f'MatchStats{league.upper()}27.xlsx'
FILE_RE = re.compile(re.escape(JSON_MASK).replace(r'\*', r'(\d+)'))

# Подключаем Google Drive и выбираем базу лиги
drive.mount('/content/drive')
DB_NAME = 'rpl.db' if league == 'rpl' else 'FA_Premier_League.db'
DB_PATH = Path('/content/drive/MyDrive/Data_Bases') / DB_NAME
if not DB_PATH.is_file():
    raise FileNotFoundError(f'Не найдена база данных: {DB_PATH}')

# JSON: составные показатели "успешно/всего"
SPLIT_STATS = {
    'Long balls': ('Long balls', 'Accurate long balls'),
    'Crosses': ('Crosses', 'Accurate crosses'),
    'Ground duels': ('Ground duels', 'Ground duels won'),
    'Aerial duels': ('Aerial duels', 'Aerial duels won'),
    'Dribbles': ('Dribbles', 'Dribbles successful'),
    'Final third phase': ('Final third phase', 'Successful final third phase'),
}

# Итоговый столбец Excel -> точное имя показателя в JSON
STAT_MAPPING = {
    'corners': 'Corner kicks',
    'yellow_cards': 'Yellow cards',
    'red_cards': 'Red cards',
    'offsides': 'Offsides',
    'fouls': 'Fouls',
    'saves': 'Goalkeeper saves',
    'possession': 'Ball possession',
    'shots': 'Total shots',
    'shots_on_target': 'Shots on target',
    'hit_woodwork': 'Hit woodwork',
    'shots_off_target': 'Shots off target',
    'blocked_shots': 'Blocked shots',
    'shots_inside_box': 'Shots inside box',
    'shots_outside_box': 'Shots outside box',
    'passes': 'Passes',
    'accurate_passes': 'Accurate passes',
    'final_third_phase': 'Final third phase',
    'accurate_final_third_phase': 'Successful final third phase',
    'long_balls': 'Long balls',
    'accurate_long_balls': 'Accurate long balls',
    'crosses': 'Crosses',
    'accurate_crosses': 'Accurate crosses',
    'ground_duels': 'Ground duels',
    'ground_duels_won': 'Ground duels won',
    'aerial_duels': 'Aerial duels',
    'aerial_duels_won': 'Aerial duels won',
    'dribbles': 'Dribbles',
    'dribbles_succ': 'Dribbles successful',
    'throw_ins': 'Throw-ins',
    'through_balls': 'Through balls',
    'distance_covered': 'Distance covered',
    'number_of_sprints': 'Number of sprints',
    'touches_in_opp_box': 'Touches in penalty area',
    'final_third_entries': 'Final third entries',  # отдельный показатель, не Final third phase
    'fouled_in_final_third': 'Fouled in final third',
}

COLUMNS = [
    'match_id', 'team_id', 'goals', 'xg', 'ppda', 'deep', 'corners',
    'yellow_cards', 'red_cards', 'offsides', 'fouls', 'saves', 'possession',
    'shots', 'shots_on_target', 'hit_woodwork', 'shots_off_target',
    'blocked_shots', 'shots_inside_box', 'shots_outside_box', 'passes',
    'accurate_passes', 'final_third_phase', 'accurate_final_third_phase',
    'long_balls', 'accurate_long_balls', 'crosses', 'accurate_crosses',
    'ground_duels', 'ground_duels_won', 'aerial_duels', 'aerial_duels_won',
    'dribbles', 'dribbles_succ', 'throw_ins', 'through_balls',
    'distance_covered', 'number_of_sprints', 'touches_in_opp_box',
    'final_third_entries', 'fouled_in_final_third',
]


def parse_value(raw, numeric):
    """Как в предыдущем скрипте: сначала homeValue/awayValue, затем raw."""
    if numeric is not None:
        return numeric
    if raw is None:
        return None
    try:
        return int(raw)
    except (TypeError, ValueError):
        try:
            return float(raw)
        except (TypeError, ValueError):
            return raw


def parse_split(raw, numeric):
    """Из 'успешно/всего' получает (всего, успешно), как исходный скрипт."""
    successful = total = None
    if isinstance(raw, str):
        found = re.search(r'(\d+)\s*/\s*(\d+)', raw)
        if found:
            successful, total = map(int, found.groups())
        else:
            found = re.search(r'\d+', raw)
            if found:
                total = int(found.group())
    if total is None and numeric is not None:
        total = numeric
    if successful is None and total is not None:
        successful = 0
    return total, successful


def get_team_ids(db, match_id):
    row = db.execute(
        'SELECT team1_id, team2_id FROM Matches WHERE match_id = ?',
        (match_id,),
    ).fetchone()
    if row is None:
        raise ValueError(f'match_id={match_id} отсутствует в Matches')
    return row  # team1_id = Home, team2_id = Away


def read_match(json_path, match_id, team_ids):
    with json_path.open(encoding='utf-8') as file:
        data = json.load(file)

    aggregated = {}
    for period in data.get('statistics', []):
        if period.get('period') != 'ALL':
            continue
        for group in period.get('groups', []):
            for item in group.get('statisticsItems', []):
                name = item.get('name')
                if not name:
                    continue
                for side in ('home', 'away'):
                    raw, numeric = item.get(side), item.get(side + 'Value')
                    if name in SPLIT_STATS:
                        total_name, successful_name = SPLIT_STATS[name]
                        total, successful = parse_split(raw, numeric)
                        aggregated[f'{side}_{total_name}'] = total
                        aggregated[f'{side}_{successful_name}'] = successful
                    else:
                        value = parse_value(raw, numeric)
                        if value is not None:
                            aggregated[f'{side}_{name}'] = value
        break  # обрабатываем только период ALL

    if not aggregated:
        print(f'{json_path.name}: нет статистики периода ALL — пропуск')
        return []

    rows = []
    for side, team_id in zip(('home', 'away'), team_ids):
        row = {column: None for column in COLUMNS}
        row.update({'match_id': match_id, 'team_id': team_id})
        for excel_column, json_name in STAT_MAPPING.items():
            row[excel_column] = aggregated.get(f'{side}_{json_name}')
        rows.append(row)
    return rows


json_files = sorted(
    (p for p in Path('.').glob(JSON_MASK) if p.is_file()),
    key=lambda p: int(FILE_RE.fullmatch(p.name).group(1)),
)
if not json_files:
    raise SystemExit(f"Файлы '{JSON_MASK}' не найдены в текущей папке.")

all_rows = []
with sqlite3.connect(str(DB_PATH)) as db:
    for json_path in json_files:
        match_id = int(FILE_RE.fullmatch(json_path.name).group(1))
        try:
            team_ids = get_team_ids(db, match_id)
            rows = read_match(json_path, match_id, team_ids)
            all_rows.extend(rows)
            if rows:
                print(f'{json_path.name}: Home={team_ids[0]}, Away={team_ids[1]}')
        except Exception as error:
            print(f'Ошибка в {json_path.name}: {error}')

if all_rows:
    result = pd.DataFrame(all_rows).reindex(columns=COLUMNS)
    # Порядок в каждом матче: Home, затем Away (не зависит от team_id)
    result['_side_order'] = result.groupby('match_id').cumcount()
    result = result.sort_values(['match_id', '_side_order'], kind='stable')
    result = result.drop(columns='_side_order').reset_index(drop=True)
    result.to_excel(OUTPUT, index=False, engine='openpyxl')
    print(f'Готово: обработано {len(all_rows) // 2} матчей, файл {OUTPUT}')
else:
    print('Нет данных для сохранения.')


Mounted at /content/drive
ms_rpl_585.json: Home=6, Away=8
ms_rpl_586.json: Home=15, Away=5
ms_rpl_587.json: Home=14, Away=20
ms_rpl_588.json: Home=2, Away=1
ms_rpl_589.json: Home=10, Away=16
ms_rpl_590.json: Home=4, Away=9
ms_rpl_591.json: Home=2, Away=7
ms_rpl_592.json: Home=17, Away=13
Готово: обработано 8 матчей, файл MatchStatsRPL27.xlsx
